<a href="https://colab.research.google.com/github/jakob-steuer/chemoinformatics_course/blob/main/chemoinformatics_6_pharmacophore_qsar_nlp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pharmacophores, QSAR and AI-assisted reading

Three ways of answering the same question — *what makes a molecule work?*

| part | question | approach |
|---|---|---|
| **1 — Pharmacophores** | which features does a molecule need? | chemical reasoning, by hand |
| **2 — QSAR** | can a model learn those features from data? | machine learning |
| **3 — Chemical space** | what does our chemistry look like as a whole? | fingerprints + UMAP |
| **4 — Reading machines** | can a model pull facts out of a paper? | NLP question answering |

Parts 1 and 4 feed directly into your semester project, and the last section of Part 1 is
written for **your own drug–target pair**.

In [ ]:
# Install the packages Colab does not ship with (about a minute).
%pip install -q rdkit umap-learn py3Dmol networkx

In [ ]:
import os
import itertools
import collections
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
from tqdm.auto import tqdm

from rdkit import Chem, RDConfig, RDLogger, DataStructs
from rdkit.Chem import AllChem, Descriptors, Draw, ChemicalFeatures
from rdkit.Chem.Scaffolds import MurckoScaffold
from rdkit.Chem.rdFingerprintGenerator import GetMorganGenerator
import py3Dmol

warnings.filterwarnings("ignore")
RDLogger.DisableLog("rdApp.*")        # RDKit complains loudly about every unparsable SMILES
plt.rcParams.update({"figure.figsize": (9, 5), "axes.grid": True, "grid.alpha": 0.3,
                     "font.size": 11, "figure.dpi": 110})

print("RDKit ready.")

# Part 1 — Pharmacophores

## What a pharmacophore is

The IUPAC definition:

> A pharmacophore is the **ensemble of steric and electronic features** that is necessary to
> ensure the optimal supramolecular interactions with a specific biological target and to trigger
> (or block) its biological response.

Three things follow, and they are the whole of this section.

**1. It is not a molecule but a pattern** — "a donor here, an aromatic ring 5 Å away, a positive
charge 7 Å from that". Any molecule that can present those features in that arrangement is a
candidate, which is why pharmacophores let you **scaffold-hop**.

**2. It is 3D.** The right features in the wrong geometry do not bind, and geometry needs a
conformer — so a flexible molecule has to be checked in many conformations.

**3. It is a hypothesis**, derived from known actives or from a complex structure, and no better
than that evidence.

### The standard feature types

| feature | meaning | typical chemistry |
|---|---|---|
| 🔵 **Donor** | hydrogen bond donor | –OH, –NH, –NH₂ |
| 🔴 **Acceptor** | hydrogen bond acceptor | C=O, ring N, ether O |
| 🟣 **Aromatic** | aromatic ring (π-stacking) | benzene, pyridine, indole |
| 🟢 **PosIonizable** | positively charged at pH 7 | basic amine, guanidine, amidine |
| 🟠 **NegIonizable** | negatively charged at pH 7 | carboxylate, phosphate, tetrazole |
| 🟡 **Hydrophobe** | greasy contact surface | alkyl chains, halogens, rings |

RDKit assigns all of these automatically. That is where we start.

## The pharmacophore toolbox

Run this cell once. Everything in Part 1 is built from these helpers, so that the exercises stay
about chemistry rather than about code.

In [ ]:
# @title
# --- feature definitions shipped with RDKit -------------------------------
FEATURE_FACTORY = ChemicalFeatures.BuildFeatureFactory(
    os.path.join(RDConfig.RDDataDir, "BaseFeatures.fdef"))

FAMILIES = ["Donor", "Acceptor", "Aromatic", "PosIonizable", "NegIonizable", "Hydrophobe"]
DEFAULT_SHOWN = ("Donor", "Acceptor", "Aromatic", "PosIonizable", "NegIonizable")
COLORS = {"Aromatic": (0.72, 0.55, 0.90), "Donor": (0.35, 0.65, 0.98),
          "Acceptor": (0.95, 0.40, 0.40), "PosIonizable": (0.40, 0.80, 0.45),
          "NegIonizable": (0.98, 0.60, 0.25), "Hydrophobe": (0.98, 0.85, 0.35)}
COLORS["LumpedHydrophobe"] = COLORS["Hydrophobe"]
HEX = {k: "#%02x%02x%02x" % tuple(int(255 * c) for c in v) for k, v in COLORS.items()}

# An atom can belong to several families at once (a basic NH is both a Donor and a
# PosIonizable). A 2D picture has only one colour per atom, so we need a precedence:
# the family furthest to the right wins.
DRAW_ORDER = ["LumpedHydrophobe", "Hydrophobe", "Aromatic", "Acceptor", "Donor",
              "NegIonizable", "PosIonizable"]


def mol_from(smiles_or_mol):
    """Accept a SMILES string or an RDKit molecule."""
    if isinstance(smiles_or_mol, str):
        return Chem.MolFromSmiles(smiles_or_mol)
    return smiles_or_mol


def smiles_from_pubchem(name):
    """Look up a compound by name in PubChem and return its SMILES (None if not found)."""
    url = ("https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/"
           f"{requests.utils.quote(name)}/property/CanonicalSMILES/TXT")
    try:
        answer = requests.get(url, timeout=20)
        if answer.ok and answer.text.strip():
            return answer.text.strip().split("\n")[0]
    except Exception as error:
        print(f"  PubChem lookup failed for {name!r}: {error}")
    return None


def feature_table(mol):
    """Every pharmacophore feature RDKit finds, as a table."""
    mol = mol_from(mol)
    rows = [{"family": f.GetFamily(), "definition": f.GetType(),
             "atoms": ", ".join(str(a) for a in f.GetAtomIds())}
            for f in FEATURE_FACTORY.GetFeaturesForMol(mol) if f.GetFamily() in FAMILIES]
    return pd.DataFrame(rows)


def feature_counts(named_molecules):
    """Feature profile of several molecules: {name: SMILES or mol} -> table."""
    rows = {}
    for name, mol in named_molecules.items():
        mol = mol_from(mol)
        counts = dict.fromkeys(FAMILIES, 0)
        for f in FEATURE_FACTORY.GetFeaturesForMol(mol):
            if f.GetFamily() in counts:
                counts[f.GetFamily()] += 1
        counts["heavy atoms"] = mol.GetNumHeavyAtoms()
        counts["MW"] = round(Descriptors.MolWt(mol), 1)
        counts["LogP"] = round(Descriptors.MolLogP(mol), 2)
        rows[name] = counts
    return pd.DataFrame(rows).T


def show_features(mol, families=DEFAULT_SHOWN, size=(900, 450), legend="", overlap="precedence"):
    """2D structure with the pharmacophore features highlighted in the colours of the table.

    overlap : how to draw an atom that belongs to several families
              "precedence" (default) — one colour, the strongest family wins (see DRAW_ORDER)
              "rings"                — one coloured ring per family, drawn concentrically
    """
    from IPython.display import SVG
    mol = mol_from(mol)
    feats = [f for f in FEATURE_FACTORY.GetFeaturesForMol(mol) if f.GetFamily() in families]
    drawer = Draw.rdMolDraw2D.MolDraw2DSVG(*size)

    if overlap == "rings":
        colour_map = collections.defaultdict(list)
        for f in feats:
            for atom in f.GetAtomIds():
                colour = COLORS[f.GetFamily()]
                if colour not in colour_map[atom]:
                    colour_map[atom].append(colour)
        drawer.DrawMoleculeWithHighlights(
            Draw.rdMolDraw2D.PrepareMolForDrawing(mol), legend, dict(colour_map), {},
            {a: 0.45 for a in colour_map}, {a: 1 for a in colour_map})
    else:
        feats.sort(key=lambda f: DRAW_ORDER.index(f.GetFamily()))
        atoms, colors, radii = [], {}, {}
        for f in feats:
            for atom in f.GetAtomIds():
                atoms.append(atom)
                colors[atom] = COLORS[f.GetFamily()]      # later families overwrite earlier ones
                radii[atom] = 0.45
        Draw.rdMolDraw2D.PrepareAndDrawMolecule(
            drawer, mol, legend=legend, highlightAtoms=list(dict.fromkeys(atoms)),
            highlightAtomColors=colors, highlightAtomRadii=radii)

    drawer.FinishDrawing()
    return SVG(drawer.GetDrawingText())


def feature_overlaps(mol, families=DEFAULT_SHOWN):
    """Atoms that belong to more than one feature family, and which colour they end up with."""
    mol = mol_from(mol)
    roles = collections.defaultdict(set)
    for f in FEATURE_FACTORY.GetFeaturesForMol(mol):
        if f.GetFamily() in families:
            for atom in f.GetAtomIds():
                roles[atom].add(f.GetFamily())
    rows = [{"atom": atom,
             "element": mol.GetAtomWithIdx(atom).GetSymbol(),
             "H": mol.GetAtomWithIdx(atom).GetTotalNumHs(),
             "aromatic": mol.GetAtomWithIdx(atom).GetIsAromatic(),
             "roles": " + ".join(sorted(families_here)),
             "drawn as": max(families_here, key=DRAW_ORDER.index)}
            for atom, families_here in sorted(roles.items()) if len(families_here) > 1]
    return pd.DataFrame(rows)


def make_conformers(mol, n_conformers=10, seed=0xF00D):
    """Generate and minimise 3D conformers — a pharmacophore only exists in 3D."""
    mol = Chem.AddHs(Chem.Mol(mol_from(mol)))
    if not AllChem.EmbedMultipleConfs(mol, numConfs=n_conformers, randomSeed=seed):
        return None
    AllChem.MMFFOptimizeMoleculeConfs(mol, maxIters=400)
    return mol


def _features_of(mol3d, conf_id, family):
    """(position, atom indices) for every feature of one family in one conformer."""
    found = []
    for f in FEATURE_FACTORY.GetFeaturesForMol(mol3d, confId=conf_id):
        if f.GetFamily() == family:
            position = f.GetPos(conf_id)
            found.append((np.array([position.x, position.y, position.z]),
                          tuple(f.GetAtomIds())))
    return found


def _feature_positions(mol3d, conf_id, family):
    return [position for position, _ in _features_of(mol3d, conf_id, family)]


def feature_distance_range(mol3d, family_a, family_b):
    """Shortest and longest distance between two feature types, over all conformers."""
    found = []
    for conf_id in range(mol3d.GetNumConformers()):
        for pa, pb in itertools.product(_feature_positions(mol3d, conf_id, family_a),
                                        _feature_positions(mol3d, conf_id, family_b)):
            if not np.allclose(pa, pb):
                found.append(float(np.linalg.norm(pa - pb)))
    return (min(found), max(found)) if found else (None, None)


def matches_pharmacophore(mol3d, points, distances, all_matches=False):
    """Does any conformer satisfy all constraints at once?

    points      : feature families, e.g. ["PosIonizable", "Aromatic", "Acceptor"]
    distances   : {(i, j): (min, max)} in Å, indices refer to `points`
    all_matches : False -> (matched, first match found)
                  True  -> a list of every distinct way the query can be satisfied

    A match is a dict with the conformer, the position and the **atoms** of each point,
    and the distances between them.
    """
    collected, seen = [], set()
    if mol3d is None:
        return [] if all_matches else (False, None)

    for conf_id in range(mol3d.GetNumConformers()):
        candidates = [_features_of(mol3d, conf_id, family) for family in points]
        if any(len(c) == 0 for c in candidates):
            continue
        for combination in itertools.product(*candidates):
            found = {}
            for (i, j), (low, high) in distances.items():
                distance = float(np.linalg.norm(combination[i][0] - combination[j][0]))
                if not low <= distance <= high:
                    break
                found[(i, j)] = round(distance, 1)
            else:
                match = {"conformer": conf_id,
                         "positions": [c[0] for c in combination],
                         "atoms": [c[1] for c in combination],
                         "distances": found}
                if not all_matches:
                    return True, match
                key = tuple(match["atoms"])
                if key not in seen:
                    seen.add(key)
                    collected.append(match)
    return collected if all_matches else (False, None)


def describe_group(mol, atom_ids):
    """A short chemical name for the group a feature sits on."""
    atom = mol.GetAtomWithIdx(atom_ids[0])
    if len(atom_ids) > 1:
        return f"ring of {len(atom_ids)} atoms"
    neighbours = list(atom.GetNeighbors())
    has_hydrogen = any(n.GetSymbol() == "H" for n in neighbours)
    next_to_ring = any(n.GetIsAromatic() for n in neighbours)
    if atom.GetSymbol() == "O":
        if any(b.GetBondTypeAsDouble() == 2 for b in atom.GetBonds()):
            carbon = [n for n in neighbours if n.GetSymbol() == "C"][0]
            others = [n.GetSymbol() for n in carbon.GetNeighbors() if n.GetIdx() != atom_ids[0]]
            if "O" in others:
                return "ester C=O"
            return "amide C=O" if "N" in others else "ketone C=O"
        if has_hydrogen:
            return "phenol -OH" if next_to_ring else "alcohol -OH"
        return "ether -O-"
    if atom.GetSymbol() == "N":
        return f"{'aromatic ' if atom.GetIsAromatic() else ''}N with {atom.GetTotalNumHs()} H"
    return atom.GetSymbol()


def match_report(named_molecules, points, distances, n_conformers=10):
    """Which molecules match a query — and which chemical group satisfied each point."""
    rows = []
    for name, molecule in named_molecules.items():
        mol3d = make_conformers(molecule, n_conformers=n_conformers)
        matched, match = matches_pharmacophore(mol3d, points, distances)
        row = {"molecule": name, "matches": matched}
        if matched:
            for family, atom_ids in zip(points, match["atoms"]):
                row[family] = describe_group(mol3d, atom_ids)
            for (i, j), distance in match["distances"].items():
                row[f"{points[i][:4]}-{points[j][:4]} (A)"] = distance
        rows.append(row)
    return pd.DataFrame(rows).fillna("")


# --- 3D viewing ----------------------------------------------------------
def _marker_model(view, positions, colour, radius, opacity=1.0, target=None):
    """Draw spheres at the given positions."""
    if len(positions) == 0:
        return
    where = {} if target is None else {"viewer": target}
    xyz = f"{len(positions)}\n\n" + "\n".join(
        f"He {p[0]:.4f} {p[1]:.4f} {p[2]:.4f}" for p in positions)
    view.addModel(xyz, "xyz", **where)
    view.setStyle({"model": -1},
                  {"sphere": {"radius": radius, "color": colour, "opacity": opacity}}, **where)


def _add_molecule(view, mol3d, conf_id=0, target=None):
    where = {} if target is None else {"viewer": target}
    view.addModel(Chem.MolToMolBlock(mol3d, confId=conf_id), "sdf", **where)
    view.setStyle({"model": -1},
                  {"stick": {"radius": 0.13, "colorscheme": "lightgreyCarbon"}}, **where)


def _dotted_line(view, start, end, target=None, colour="#333333", n_dots=11, radius=0.09):
    """Draw a dotted line between two points."""
    _marker_model(view, [start + (end - start) * t for t in np.linspace(0.1, 0.9, n_dots)],
                  colour, radius, 1.0, target)


def view_features_3d(mol, families=DEFAULT_SHOWN, width=760, height=420, zoom=1.6):
    """One conformer with a coloured sphere on every pharmacophore feature."""
    mol3d = make_conformers(mol, n_conformers=1)
    view = py3Dmol.view(width=width, height=height)
    _add_molecule(view, mol3d)
    for family in families:
        _marker_model(view, _feature_positions(mol3d, 0, family), HEX[family], 0.85, 0.80)
    view.setBackgroundColor("white")
    view.zoomTo()
    view.zoom(zoom)                     # zoomTo fits the bounding sphere, which leaves a long
    return view                         # molecule looking tiny; tighten it up


def view_pharmacophore(mol, points, distances, label="", n_conformers=10,
                       width=760, height=460, view=None, target=None, zoom=1.35):
    """Show the conformer that satisfies a pharmacophore query.

    The matching features are drawn as spheres and the distances between them as dotted
    lines. Pass `view` and `target=(row, column)` to draw into one cell of a viewer grid.
    """
    where = {} if target is None else {"viewer": target}
    mol3d = make_conformers(mol, n_conformers=n_conformers)
    matched, match = matches_pharmacophore(mol3d, points, distances)
    if view is None:
        view = py3Dmol.view(width=width, height=height)

    _add_molecule(view, mol3d, match["conformer"] if matched else 0, target)
    if matched:
        for family, position in zip(points, match["positions"]):
            _marker_model(view, [position], HEX[family], 0.9, 0.80, target)
        for (i, j), distance in match["distances"].items():
            start, end = match["positions"][i], match["positions"][j]
            _dotted_line(view, start, end, target)
            middle = (start + end) / 2
            view.addLabel(f"{distance} A",
                          {"position": {"x": float(middle[0]), "y": float(middle[1]),
                                        "z": float(middle[2])},
                           "fontSize": 11, "fontColor": "black", "backgroundColor": "white",
                           "backgroundOpacity": 0.75, "borderThickness": 0}, **where)
    if label:
        # the leading spaces keep the label clear of the edge of the viewer
        view.addLabel(f"  {label}" if matched else f"  {label}: no match",
                      {"useScreen": True, "x": 10, "y": 10, "fontSize": 14,
                       "fontColor": "black" if matched else "white",
                       "backgroundColor": "white" if matched else "#cc3333",
                       "backgroundOpacity": 0.85}, **where)
    view.setBackgroundColor("white")
    view.zoomTo(**where)
    view.zoom(zoom, **where)
    return view


print("Pharmacophore toolbox ready.")

## Step 1 — Which features does a molecule have?

Let us take **imatinib** (Gleevec), the drug that turned chronic myeloid leukaemia from a fatal
disease into a managed one. The colours are those of the table above.

In [ ]:
IMATINIB = "CC1=C(C=C(C=C1)NC(=O)C2=CC=C(C=C2)CN3CCN(CC3)C)NC4=NC=CC(=N4)C5=CN=CC=C5"

show_features(IMATINIB, legend="imatinib")

In [ ]:
feature_table(IMATINIB).groupby("family").size().to_frame("count")

Read the picture, not the table: the two **blue** NH groups and the **red** ring nitrogens are the
hinge-binding motif that every kinase inhibitor needs; the **purple** rings stack in a mostly
greasy pocket; the **green** piperazine nitrogens are protonated at pH 7 and reach out to solvent,
which is largely why the molecule is soluble enough to be a tablet.

That sentence is a pharmacophore hypothesis. The rest of Part 1 makes it testable.

### One atom, several roles

An atom can belong to several families at once, and a 2D picture has only one colour per atom.
Here is where that happens in imatinib:

In [ ]:
feature_overlaps(IMATINIB)

Two cases, and the second is worth knowing:

* **Ring nitrogens are Acceptor + Aromatic** — both true, and independent of each other.
* **The piperazine nitrogens are Donor + PosIonizable with no hydrogen at all** (`H` column:
  zero). RDKit lists trialkylamines as donors deliberately: such a nitrogen is protonated at
  pH 7, and the resulting N–H⁺ *is* a donor. For a basic amine the two labels are not a
  coincidence — they are the same chemistry described twice.

The drawing breaks the tie by precedence, strongest last:

`Hydrophobe → Aromatic → Acceptor → Donor → NegIonizable → PosIonizable`

so a basic NH is drawn green and a pyridine N red, with the rest of its ring still purple.
Use `overlap="rings"` to see every role at once:

In [ ]:
show_features(IMATINIB, overlap="rings", legend="imatinib — every role drawn")

More honest, less legible — the rest of the notebook keeps the single-colour version. The
**numbers** lose nothing: `feature_table` and `feature_counts` count features, not atoms, so a
basic NH is counted under both families. Only the picture has to choose.

### Exercise 1 — Features of a drug you know

Pick any drug and look at its features. `smiles_from_pubchem` fetches the structure by name — the
same PubChem API you will use in the project. Try deliberately different molecules: an antibiotic,
a steroid, a sugar, your project compound.

<details>
<summary>🔍 Something to notice</summary>

```python
for name in ["aspirin", "caffeine", "atorvastatin", "penicillin G", "glucose"]:
    display(show_features(smiles_from_pubchem(name), legend=name))
```

* **Aspirin** has an 🟠 ionizable acid — an anion at pH 7.4, which is why it is absorbed in the
  stomach (where it is neutral) rather than the gut.
* **Glucose** is nothing but 🔵 donors and 🔴 acceptors: no ring, no hydrophobe, no charge. It
  still crosses the blood–brain barrier — through a dedicated transporter, which a features-only
  view cannot know about. Keep that in mind for Part 2.
* **Atorvastatin** shows how large a real drug's feature set is, and how little a five-descriptor
  model captures.

</details>

## Step 2 — The same features in 3D

Features alone are not a pharmacophore. What matters is **where they are relative to each other**,
and that only exists once the molecule has a conformation.

Drag to rotate: every sphere is one feature, coloured as in the table.

In [ ]:
view_features_3d(IMATINIB)

Now we can measure the thing that makes a pharmacophore a *model*: the distance between features.
Because the molecule is flexible, we generate several conformers and report the range.

In [ ]:
imatinib_3d = make_conformers(IMATINIB, n_conformers=10)

print(f"imatinib, {imatinib_3d.GetNumConformers()} conformers\n")
for family_a, family_b in [("Donor", "Aromatic"), ("Donor", "Acceptor"),
                           ("PosIonizable", "Aromatic"), ("PosIonizable", "Acceptor")]:
    low, high = feature_distance_range(imatinib_3d, family_a, family_b)
    print(f"{family_a:>13s} ... {family_b:<13s} {low:5.1f} – {high:5.1f} Å")

A flexible molecule covers a wide range of distances — imatinib can place its basic nitrogen
anywhere between roughly 4 and 19 Å from an aromatic ring. That flexibility is what lets one molecule
satisfy many different pharmacophores, and it is also why a pharmacophore search must be done over
conformers rather than on a single structure.

## Step 3 — Scaffold hopping: the point of the whole idea

Here is the classic demonstration. These four molecules are all strong **µ-opioid receptor
agonists** — they are painkillers that act on the same receptor:

* **morphine** — the natural product, a rigid five-ring cage
* **pethidine** (meperidine) — a simple piperidine ester
* **methadone** — an open chain, no ring system at all connecting the parts
* **fentanyl** — a different open scaffold again

If you compare them the way a chemist's fingerprint does — by substructure — they have almost
nothing in common. Let us check that first.

In [ ]:
OPIOIDS = {
    "morphine":  "CN1CCC23C4C1CC5=C2C(=C(C=C5)O)OC3C(C=C4)O",
    "pethidine": "CCOC(=O)C1(CCN(CC1)C)C2=CC=CC=C2",
    "methadone": "CCC(=O)C(CC(C)N(C)C)(C1=CC=CC=C1)C2=CC=CC=C2",
    "fentanyl":  "CCC(=O)N(C1CCN(CC1)CCC2=CC=CC=C2)C3=CC=CC=C3",
}
CONTROLS = {
    "aspirin":  "CC(=O)OC1=CC=CC=C1C(=O)O",
    "caffeine": "CN1C=NC2=C1C(=O)N(C(=O)N2C)C",
    "glucose":  "C(C1C(C(C(C(O1)O)O)O)O)O",
}

morgan = GetMorganGenerator(radius=2, fpSize=2048)
reference = morgan.GetFingerprint(Chem.MolFromSmiles(OPIOIDS["morphine"]))

print("Morgan (substructure) similarity to morphine:")
for name, smiles in {**OPIOIDS, **CONTROLS}.items():
    similarity = DataStructs.TanimotoSimilarity(
        reference, morgan.GetFingerprint(Chem.MolFromSmiles(smiles)))
    print(f"   {name:10s} {similarity:.2f}")

Draw.MolsToGridImage([Chem.MolFromSmiles(s) for s in OPIOIDS.values()],
                     legends=list(OPIOIDS), molsPerRow=4, subImgSize=(260, 220))

Now ask the pharmacophore question instead. Since the 1950s the classical model for this family —
the **Beckett–Casy** model — has been two points plus a geometric relation:

* a **protonated nitrogen** (🟢 PosIonizable), which makes a salt bridge with a conserved
  aspartate in the receptor,
* a **flat aromatic ring** (🟣),
* separated by roughly **4–7 Å**.

That is all. Two points and one distance window:

In [ ]:
OPIOID_PHARMACOPHORE = (
    ["PosIonizable", "Aromatic"],        # the two points
    {(0, 1): (4.0, 7.0)},                # N+ ... ring centre, in Å
)

Here it is on morphine. The spheres are the two pharmacophore points and the dotted line is the
distance that satisfies the query — this picture *is* the pharmacophore:

In [ ]:
view_pharmacophore(OPIOIDS["morphine"], *OPIOID_PHARMACOPHORE, label="morphine")

Note what is **not** in that picture: no carbon skeleton, no ring system, no stereochemistry. A
pharmacophore keeps the coloured points and the distances between them and throws the rest of the
molecule away. So any other molecule that can arrange the same points the same way is a candidate —
whatever it is built from.

Let us test that against the four opioids and the three controls. `match_report` also names the
chemical group that satisfied each point, which we will need in a moment.

In [ ]:
match_report({**OPIOIDS, **CONTROLS}, *OPIOID_PHARMACOPHORE)

All four opioids match, at 4.7, 5.7, 6.1 and 5.2 Å. None of the controls does: aspirin and glucose
have no basic nitrogen at all, and caffeine's nitrogens sit *inside* its rings rather than 4–7 Å
away from one.

Here they are side by side — the same two points, four unrelated scaffolds:

In [ ]:
grid = py3Dmol.view(viewergrid=(2, 2), width=860, height=620, linked=False)

for (name, smiles), cell in zip(OPIOIDS.items(), [(0, 0), (0, 1), (1, 0), (1, 1)]):
    view_pharmacophore(smiles, *OPIOID_PHARMACOPHORE, label=name, view=grid, target=cell)

grid.setBackgroundColor("white")
grid

Rotate any of the four and compare. The molecules have almost nothing in common — morphine is a
rigid five-ring cage, methadone an open chain — but each presents a basic nitrogen and an aromatic
ring at the same separation. Morphine does it without effort; the flexible ones (fentanyl,
methadone) must adopt a particular conformation to manage it, which costs energy and is one reason
their potencies differ.

This is the single most important idea in Part 1: **a pharmacophore finds molecules that a
substructure search cannot.** Fentanyl is a 0.13-similarity stranger to morphine and yet it
presents the same two features in the same arrangement — which is why it binds the same receptor
(and, being far more potent, why it is so dangerous).

### Exercise 2 — Would a third point be better?

Two points is a thin model. All four molecules also contain a **hydrogen bond acceptor**, so the
obvious improvement is to add it as a third point:

```python
THREE_POINT = (
    ["PosIonizable", "Aromatic", "Acceptor"],
    {(0, 1): (4.0, 7.0),      # N+ ... ring centre
     (0, 2): (4.0, 9.0),      # N+ ... acceptor
     (1, 2): (2.5, 7.0)},     # ring centre ... acceptor
)
```

Run `match_report(OPIOIDS, *THREE_POINT)` and read it **column by column** before going on. Do all
four still match — and did the third point land on the same kind of group each time?

<details>
<summary>🔍 Stage 1 — all four match, and that is the problem</summary>

```python
THREE_POINT = (
    ["PosIonizable", "Aromatic", "Acceptor"],
    {(0, 1): (4.0, 7.0), (0, 2): (4.0, 9.0), (1, 2): (2.5, 7.0)},
)

match_report(OPIOIDS, *THREE_POINT)
```

| molecule | Acceptor | N⁺–acc (Å) | ring–acc (Å) |
|---|---|---|---|
| morphine | **phenol –OH** | 7.2 | **2.8** |
| pethidine | **ether –O–** | 5.0 | 4.0 |
| methadone | **ketone C=O** | 4.6 | 4.6 |
| fentanyl | **amide C=O** | 4.7 | 5.0 |

All four match, and the model is nonsense: the third point is satisfied by a phenol, an ether, a
ketone and an amide. Morphine gives it away numerically — its acceptor sits 2.8 Å from the ring
centre (a substituent *on* the ring) and 7.2 Å from the nitrogen, where the others sit 4–5 Å from
both.

A query that "works" because each molecule found its own unrelated way to satisfy it has told you
nothing. **Always look at which atoms matched, not just whether something matched.**

</details>

<details>
<summary>🔍 Stage 2 — tighten the windows and it becomes consistent</summary>

The numbers suggest the fix: exclude acceptors sitting on the ring, and require a similar distance
to both other points.

```python
THREE_POINT_TIGHT = (
    ["PosIonizable", "Aromatic", "Acceptor"],
    {(0, 1): (4.0, 7.0), (0, 2): (4.5, 6.5), (1, 2): (3.5, 6.0)},
)

match_report({**OPIOIDS, **CONTROLS}, *THREE_POINT_TIGHT)
```

Now the assignments line up — morphine through its **C6 alcohol**, methadone and fentanyl through
their **carbonyls**, pethidine through its ester — and the controls still fail.

Except for one detail, already visible in the table: pethidine is reported as matching through the
ester's **ether –O–**, not its C=O. Ask for *every* way the query can be satisfied:

```python
for name, smiles in OPIOIDS.items():
    mol3d = make_conformers(smiles, n_conformers=10)
    groups = {describe_group(mol3d, m["atoms"][2])
              for m in matches_pharmacophore(mol3d, *THREE_POINT_TIGHT, all_matches=True)}
    print(f"{name:10s} {sorted(groups)}")
```

Morphine, methadone and fentanyl have one option each. **Pethidine has two**: its ester C=O and
the ester's own –O–, about 2.2 Å apart. No distance-based query can separate two atoms that close;
only chemistry can say a carbonyl is a good acceptor and an ester's ether oxygen a poor one. That
is a hard limit of the method, not a parameter to tune.

</details>

<details>
<summary>🔍 Stage 3 — now check it against what is known</summary>

We have a model that fits all four actives, rejects the controls, and assigns a comparable group
in each molecule. It looks like a result.

Now look up what medicinal chemistry says about morphine. The hydroxyl that matters for µ-opioid
binding is the **phenol** — the one our tightened query specifically excluded. The C6 position is
the one you can acetylate (giving heroin) without losing activity.

So the model is self-consistent and still encodes the wrong interaction for one of its four
molecules. Every step was reasonable; only knowing the chemistry, or docking the ligands, would
have caught it.

Three things for your project:

1. **Matching is not evidence.** Report which atoms matched, every time.
2. **Tightening a query until it behaves means imposing what you already believed** — legitimate
   as a hypothesis, not as a finding.
3. **A pharmacophore cannot be validated against itself.** It needs a structure, a mutation or an
   assay.

Good material for your "three reasons our conclusion could be wrong" slide — and the reason the
main text sticks to the two-point model, which claims less and is true.

</details>

### Exercise 3 — How picky is our query, really?

Finding your known actives is the easy half; **not** finding everything else is the hard half. Run
the query against a few hundred ordinary drug-like molecules.

```python
library = pd.read_csv("https://deepchemdata.s3-us-west-1.amazonaws.com/datasets/BBBP.csv")
library = library.sample(150, random_state=0)
library["mol"] = library["smiles"].apply(Chem.MolFromSmiles)
library = library.dropna(subset=["mol"])

hits = 0
for mol in library["mol"]:
    # ... build conformers (3 is enough) and test the pharmacophore
    pass

print(f"{hits} / {len(library)} molecules match")
```

<details>
<summary>🔍 Solution and what it means</summary>

```python
hits = 0
for mol in library["mol"]:
    mol3d = make_conformers(mol, n_conformers=3)
    if matches_pharmacophore(mol3d, *OPIOID_PHARMACOPHORE)[0]:
        hits += 1

print(f"{hits} / {len(library)} molecules match ({hits/len(library)*100:.0f} %)")
```

About half a minute, and **56 of 148 — 38 %**. More than a third of an arbitrary drug library
satisfies our "opioid" pharmacophore. That is the central limitation of the method: most drugs
contain a ring and a basic nitrogen somewhere, flexible molecules can reach almost any distance
given enough conformers, and nothing in the query says the rest of the molecule has to **fit**.

Adding points helps, at a price:

| query | hits in 148 drugs |
|---|---|
| two-point | 38 % |
| three-point, loose windows | 22 % |
| three-point, tightened windows | 11 % |

— but Exercise 2 showed what those extra points can cost in meaning. Even at 11 % a pharmacophore
is a *filter*, not an answer. Real screening adds **excluded volumes** (regions where protein atoms
sit, so the ligand must not) and a shape constraint, then hands the survivors to docking.

</details>

## Step 4 — Your project compounds

From here on, Part 1 is your own work. Both tracks can be opened up with the tools above, and in
both cases the interesting answer is usually **what the pharmacophore does *not* explain**.

**Selectivity track** — *one drug, two related proteins; why does it bind one and not the other?*
Take a known binder of each protein and compare their feature patterns. If the two sets of actives
need different features, you have a hypothesis to check against the structures.

**Resistance track** — *wild-type versus clinical mutant; why does the drug stop working?* The drug
does not change, so start from one that still works: most clinically important resistance mutations
have a next-generation inhibitor designed to beat them. Compare the generations — what did the
chemists add, remove or rigidify?

The cell below does this for two textbook pairs.

In [ ]:
PAIRS = {
    "EGFR, T790M resistance": ["gefitinib", "osimertinib"],   # 1st vs 3rd generation
    "ABL vs SRC selectivity": ["imatinib", "dasatinib"],      # selective vs promiscuous
}

for title, drugs in PAIRS.items():
    molecules = {}
    for drug in drugs:
        smiles = smiles_from_pubchem(drug)
        if smiles:
            molecules[drug] = smiles
    if len(molecules) < 2:
        print(f"{title}: could not fetch both compounds, skipping")
        continue

    fingerprints = [morgan.GetFingerprint(Chem.MolFromSmiles(s)) for s in molecules.values()]
    similarity = DataStructs.TanimotoSimilarity(*fingerprints)
    print(f"\n### {title}  —  Morgan similarity {similarity:.2f}")
    display(feature_counts(molecules))
    display(Draw.MolsToGridImage([Chem.MolFromSmiles(s) for s in molecules.values()],
                                 legends=list(molecules), molsPerRow=2, subImgSize=(380, 300)))

### Exercise 4 — Read the comparisons

For each pair: did the number of features change much? If not, what *did* change — and what would
you need in order to explain it properly? Then run the same comparison on your own compounds and
write down a one-sentence hypothesis.

<details>
<summary>🔍 Discussion</summary>

**The features barely move.** Gefitinib and osimertinib differ by one donor and two acceptors;
imatinib and dasatinib have near-identical profiles. On feature counts alone you would call each
pair interchangeable. They are not — one member of each pair does something the other cannot.

**What changed is chemistry and geometry the feature list does not encode.**

* **Osimertinib vs gefitinib (EGFR T790M).** Osimertinib carries an **acrylamide** that forms a
  *covalent* bond to a cysteine in the pocket. No standard feature family describes "Michael
  acceptor that will react with a nearby cysteine" — the irreversible chemistry, the entire point
  of the molecule, is invisible here.
* **Dasatinib vs imatinib (selectivity).** Imatinib binds a particular *inactive* kinase
  conformation that few kinases adopt; dasatinib binds the active one, which many share. The
  selectivity is a property of the **protein's** conformational preference, not of the ligand's
  features at all.

A third, classic case you could add yourself: **ponatinib vs imatinib** for the ABL T315I
gatekeeper mutation, where the answer is a rigid carbon–carbon triple bond — a pure **shape**
change, which is exactly what a feature-count table throws away.

All three need the protein: a structure, the pocket shape, docking. That is the honest conclusion
of Part 1, and a good candidate for your "three reasons our conclusion could be wrong" slide:

> A pharmacophore is a *necessary* condition for binding, not a sufficient one. It can tell you
> why a molecule **cannot** bind. It cannot, on its own, tell you why one of two molecules with
> the same features binds better.

</details>

### Your own compounds

Edit the names (or paste SMILES) and run.

In [ ]:
MY_COMPOUNDS = ["compound_a", "compound_b"]        # <-- your drug / your two drugs

my_molecules = {}
for name in MY_COMPOUNDS:
    smiles = name if Chem.MolFromSmiles(name) else smiles_from_pubchem(name)
    if smiles is None:
        print(f"could not find {name!r} in PubChem — paste the SMILES instead")
    else:
        my_molecules[name] = smiles

display(feature_counts(my_molecules))

for name, smiles in my_molecules.items():
    display(show_features(smiles, legend=name))
    mol3d = make_conformers(smiles, n_conformers=10)
    low, high = feature_distance_range(mol3d, "Donor", "Aromatic")
    if low is not None:
        print(f"{name}: donor–aromatic distance spans {low:.1f}–{high:.1f} Å "
              f"over {mol3d.GetNumConformers()} conformers")

### Exercise 5 — Your compound in 3D

The 2D picture tells you *which* features exist; only the 3D one tells you whether a protein could
use them together. Rotate your compound and ask: which features point the same way, and which sit
on opposite faces? A pocket can only reach one side at a time.

In [ ]:
first = list(my_molecules)[0]
view_features_3d(my_molecules[first])

Then measure the distances that define your own pharmacophore hypothesis — the pairs you believe
the protein actually uses:

```python
mol3d = make_conformers(my_molecules[first], n_conformers=10)

for a, b in [("Donor", "Aromatic"), ("PosIonizable", "Acceptor")]:
    low, high = feature_distance_range(mol3d, a, b)
    if low is not None:
        print(f"{a:>13s} ... {b:<13s} {low:5.1f} – {high:5.1f} Å")
```

## Step 5 — Deriving a pharmacophore from known actives

So far we *guessed* a pharmacophore and then tested it. The real workflow runs the other way: you
have several molecules known to hit the same target and you want the model they all satisfy. That
is **pharmacophore mapping**.

The classical algorithm is **clique detection** (the method behind DISCO):

1. Take one conformer of molecule A and one of molecule B.
2. Build a **correspondence graph**. Each node is a *pair* of same-type features — "ring 2 of A
   could play the role of ring 1 of B". Two nodes are joined when they are mutually consistent:
   the distance between the features inside A matches the distance between their partners in B,
   within a tolerance.
3. A **clique** — a set of nodes all joined to each other — is a set of assignments consistent
   with one another. The **largest** clique is the biggest common pharmacophore.
4. Repeat over conformer pairs, keep the best.

Step 4 is where the cost lives: *n* molecules × *k* conformers means kⁿ combinations, and clique
detection is NP-hard. That is why **GASP** exists — a genetic algorithm that searches molecule and
conformer space together instead of enumerating it, trading the guarantee of the best answer for
the ability to handle flexible molecules and larger sets. We stay with clique detection on two
molecules, which is exact and fast enough to watch.

In [ ]:
# @title
import networkx as nx


def _correspondence_graph(features_a, features_b, tolerance, min_separation):
    """Nodes = pairs of same-type features; edges = pairs whose internal distances agree."""
    nodes = [(i, j) for i, (fa, _) in enumerate(features_a)
             for j, (fb, _) in enumerate(features_b) if fa == fb]
    graph = nx.Graph()
    graph.add_nodes_from(nodes)
    for (i, j), (k, l) in itertools.combinations(nodes, 2):
        if i == k or j == l:                       # a feature cannot play two roles
            continue
        distance_a = float(np.linalg.norm(features_a[i][1] - features_a[k][1]))
        distance_b = float(np.linalg.norm(features_b[j][1] - features_b[l][1]))
        if distance_a < min_separation or distance_b < min_separation:
            continue                               # the two points must be distinct in space
        if abs(distance_a - distance_b) <= tolerance:
            graph.add_edge((i, j), (k, l))
    return graph


def _features_with_family(mol3d, conf_id, families):
    return [(family, position) for family in families
            for position in _feature_positions(mol3d, conf_id, family)]


def common_pharmacophore(molecule_a, molecule_b, tolerance=1.5, min_separation=2.0,
                         n_conformers=5, families=DEFAULT_SHOWN):
    """Largest set of features two molecules can present in the same geometry."""
    mol_a = make_conformers(molecule_a, n_conformers=n_conformers)
    mol_b = make_conformers(molecule_b, n_conformers=n_conformers)
    best = None
    for conf_a in range(mol_a.GetNumConformers()):
        features_a = _features_with_family(mol_a, conf_a, families)
        for conf_b in range(mol_b.GetNumConformers()):
            features_b = _features_with_family(mol_b, conf_b, families)
            graph = _correspondence_graph(features_a, features_b, tolerance, min_separation)
            for clique in nx.find_cliques(graph):
                if best is None or len(clique) > len(best["clique"]):
                    best = {"clique": sorted(clique), "features_a": features_a,
                            "features_b": features_b, "conformers": (conf_a, conf_b)}
    return best


def as_query(mapping, padding=0.75):
    """Turn a common pharmacophore into a (points, distances) query."""
    clique = mapping["clique"]
    points = [mapping["features_a"][i][0] for i, _ in clique]
    distances = {}
    for (p, (i, j)), (q, (k, l)) in itertools.combinations(list(enumerate(clique)), 2):
        distance_a = float(np.linalg.norm(mapping["features_a"][i][1] - mapping["features_a"][k][1]))
        distance_b = float(np.linalg.norm(mapping["features_b"][j][1] - mapping["features_b"][l][1]))
        distances[(p, q)] = (round(min(distance_a, distance_b) - padding, 1),
                             round(max(distance_a, distance_b) + padding, 1))
    return points, distances


print("Pharmacophore mapping ready.")

Try it on two opioids that share nothing structurally — morphine and fentanyl, Tanimoto 0.13:

In [ ]:
mapping = common_pharmacophore(OPIOIDS["morphine"], OPIOIDS["fentanyl"])
derived_points, derived_distances = as_query(mapping)

print(f"{len(derived_points)} common features: {derived_points}\n")
for (p, q), (low, high) in derived_distances.items():
    print(f"  {derived_points[p]:>13s} ... {derived_points[q]:<13s} {low:4.1f} – {high:4.1f} Å")

Three points — a positive charge, an aromatic ring and an acceptor — at roughly 4–6 Å from each
other. The algorithm has **rediscovered the model we assumed in Step 3**, from the molecules alone,
without being told what to look for. That is the result you want from mapping: not a prettier
picture, but a model you did not put in by hand.

Note what it does *not* give you: any guarantee that the shared arrangement is the one the receptor
reads. Two molecules always share *something*. Exercise 2 applies here with full force.

### Exercise 6 — Map your own actives

Run `common_pharmacophore` on two compounds known to hit your target — your drug and a second
known binder, or the two generations from the resistance track.

```python
first, second = list(my_molecules)[:2]

mapping = common_pharmacophore(my_molecules[first], my_molecules[second])
my_points, my_distances = as_query(mapping)
```

Then ask:

1. How many points come out? Fewer than three and the model is too weak to screen with.
2. Are they chemically sensible for your target — does a kinase hinge binder really produce a
   donor and an acceptor a few Å apart?
3. Rerun with `tolerance=1.0` and `tolerance=2.5`. How stable is the answer? A model that changes
   completely with the tolerance is telling you the two molecules do not have much in common.

## Step 6 — Searching PubChem with your pharmacophore

A pharmacophore is worth having because you can **screen** with it. The full version of this is
virtual screening over millions of compounds; the version that fits in a notebook is a few hundred,
which is enough to hand you a short list to dock next to your own ligand.

We take the candidates from PubChem's 2D similarity search and keep the ones that satisfy the
query. Two requests and about a minute of conformer generation.

In [ ]:
def pubchem_similar(smiles, threshold=80, max_records=100, timeout=60):
    """2D-similarity search in PubChem. Returns a table of CID, name and SMILES."""
    url = ("https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/fastsimilarity_2d/smiles/"
           "property/SMILES,Title/JSON")
    answer = requests.post(url, data={"smiles": smiles, "Threshold": threshold,
                                      "MaxRecords": max_records}, timeout=timeout)
    answer.raise_for_status()
    return pd.DataFrame(answer.json()["PropertyTable"]["Properties"])


def screen(candidates, points, distances, n_conformers=3, max_heavy_atoms=60):
    """Keep the candidates whose conformers satisfy the query."""
    kept = []
    for _, row in tqdm(list(candidates.iterrows()), desc="screening", leave=False):
        mol = Chem.MolFromSmiles(row["SMILES"])
        if mol is None or mol.GetNumHeavyAtoms() > max_heavy_atoms:
            continue
        if matches_pharmacophore(make_conformers(mol, n_conformers), points, distances)[0]:
            kept.append({"CID": row["CID"], "name": row["Title"], "SMILES": row["SMILES"]})
    return pd.DataFrame(kept)


print("PubChem screening ready.")

In [ ]:
candidates = pubchem_similar(OPIOIDS["morphine"], threshold=80, max_records=60)
print(f"PubChem returned {len(candidates)} compounds")

hits = screen(candidates, derived_points, derived_distances)
print(f"{len(hits)} of them satisfy the derived pharmacophore\n")
display(hits.head(12))

Draw.MolsToGridImage([Chem.MolFromSmiles(s) for s in hits["SMILES"].head(8)],
                     legends=list(hits["name"].head(8)), molsPerRow=4, subImgSize=(250, 200))

The hit list is full of real opioids — naloxone, hydromorphone, oxycodone, buprenorphine,
etorphine — alongside a few plant alkaloids that happen to carry the same arrangement.

Be clear about what this does and does not show. We searched PubChem by **2D similarity to
morphine**, so the candidates were morphine-like before the pharmacophore saw them; the filter
confirms rather than discovers. To find genuinely new scaffolds you need a large, *diverse* library
— ChEMBL, ZINC, Enamine — and that is real virtual screening, outside this notebook. What you have
here is the practical version for the project: **a reproducible way to collect compounds related to
your ligand and narrow them to the ones that could present the same features.**

### Exercise 7 — Build your docking shortlist

Run the search for your own compound and your own derived pharmacophore:

```python
candidates = pubchem_similar(my_molecules[first], threshold=80, max_records=60)
hits = screen(candidates, my_points, my_distances)
hits.to_csv("my_pharmacophore_hits.csv", index=False)
```

Then curate it, because an automated list is not a result:

1. **Look at the structures.** Salts, isotopically labelled versions and your own compound will be
   in there.
2. **Check a few in the literature.** Are any actually known binders of your target? Known
   *inactives* are even more valuable — they are your negative controls for docking.
3. **Pick 3–5 to carry forward.** Aim for a spread: one close analogue, one that is structurally
   different, one you expect to fail.

That shortlist, with your reasons for each choice, is exactly the kind of slide that makes a
10-minute presentation defensible.

# Part 2 — QSAR: letting a model find the features

In Part 1 *we* decided which features matter, using chemical knowledge. That works when you
understand the target. When you do not — or when the property is the sum of many effects —
you can instead give a machine learning model a pile of molecules with measured values and let it
work out the relationship. That is **QSAR**: Quantitative Structure–Activity Relationship.

The property we will model is one of the hardest and most practical in drug discovery.

# QSAR Exercise: Blood-Brain Barrier

Can a drug get where it needs to go? For anything targeting the brain the answer must be yes; for
most other drugs it must be no, or you get side effects like drowsiness. The gatekeeper is the
**blood–brain barrier (BBB)**, and predicting whether a molecule crosses it is a classic ADMET
problem.

We will build a complete QSAR model: load molecules with known BBB penetration, turn them into
descriptors with RDKit, train a Random Forest, and — most importantly — work out what it learned
and where it fails.

We will follow the standard workflow — and then break it on purpose, because the way QSAR models
fail is more instructive than the way they succeed.

In [ ]:
url = "https://deepchemdata.s3-us-west-1.amazonaws.com/datasets/BBBP.csv"
df = pd.read_csv(url)

print("rows:", len(df), "| columns:", list(df.columns))
print(df["p_np"].value_counts().rename({1: "penetrates", 0: "does not"}).to_string())

df["mol"] = df["smiles"].apply(Chem.MolFromSmiles)
print(f"\nSMILES that RDKit could not parse: {df['mol'].isna().sum()}")
df = df.dropna(subset=["mol"]).reset_index(drop=True)
print("molecules left:", len(df))

**Look at the class balance before anything else.** 76 % of this dataset penetrates the barrier.
Keep that number in mind — it is the score a model gets by always answering "yes".

And before modelling anything, look at the actual molecules. In chemoinformatics, a model you
cannot trace back to structures is a model you cannot defend.

In [ ]:
penetrators = df[df["p_np"] == 1].sample(4, random_state=1)
non_penetrators = df[df["p_np"] == 0].sample(4, random_state=1)
sample = pd.concat([penetrators, non_penetrators])

Draw.MolsToGridImage(
    list(sample["mol"]), molsPerRow=4, subImgSize=(260, 220),
    legends=[f"{name[:22]}\n{'CROSSES' if p else 'does NOT cross'}"
             for name, p in zip(sample["name"], sample["p_np"])])

## Feature Engineering - The "S" in QSAR

This is the most important chemoinformatics step.

We will use RDKit to convert "pictures" (SMILES) into "numbers" (descriptors).

In [ ]:
DESCRIPTORS = ["MW", "LogP", "TPSA", "NumHDonors", "NumHAcceptors"]

df[DESCRIPTORS] = df["mol"].apply(lambda m: pd.Series([
    Descriptors.MolWt(m),           # size
    Descriptors.MolLogP(m),         # lipophilicity
    Descriptors.TPSA(m),            # polar surface area
    Descriptors.NumHDonors(m),      # H-bond donors
    Descriptors.NumHAcceptors(m),   # H-bond acceptors
]))

df[DESCRIPTORS].describe().round(1)

**Why do we use these descriptors?**

We're choosing properties related to:

- size (MW)
- polarity (TPSA, H-Donors/Acceptors)
- lipophilicity (LogP)

These are the pillars of 'druglikeness'.

Notice that four of these five descriptors are **counts or sums of pharmacophore features** —
donors, acceptors, polar surface, greasiness. Part 1 and Part 2 are asking the same question with
different tools.

## Training a model

Nothing surprising here: split the data, fit a random forest, predict the held-out part.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

X = df[DESCRIPTORS]
y = df["p_np"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

print(f"trained on {len(X_train)} molecules, testing on {len(X_test)}")

## Evaluation — and why accuracy is a trap

Accuracy on an unbalanced dataset flatters a model. The helper below reports it next to the score
of a model that has learned nothing at all and always predicts the majority class, plus three
measures that are not fooled by imbalance:

* **balanced accuracy** — the average of the two per-class recalls (50 % = coin flip),
* **ROC-AUC** — the probability that a random penetrator is ranked above a random non-penetrator,
* **MCC** — a correlation coefficient between prediction and truth (0 = worthless, 1 = perfect).

In [ ]:
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, roc_auc_score,
                             matthews_corrcoef, confusion_matrix, ConfusionMatrixDisplay)


def evaluate(model, X_test, y_test, title=""):
    predicted = model.predict(X_test)
    probability = model.predict_proba(X_test)[:, 1]
    majority = max(y_test.mean(), 1 - y_test.mean())

    print(f"--- {title} ---")
    print(f"accuracy                {accuracy_score(y_test, predicted):.3f}")
    print(f"   ... always-'yes'     {majority:.3f}   <- the score to beat")
    print(f"balanced accuracy       {balanced_accuracy_score(y_test, predicted):.3f}")
    print(f"ROC-AUC                 {roc_auc_score(y_test, probability):.3f}")
    print(f"MCC                     {matthews_corrcoef(y_test, predicted):.3f}")

    matrix = ConfusionMatrixDisplay(confusion_matrix(y_test, predicted),
                                    display_labels=["does not cross", "crosses"])
    matrix.plot(colorbar=False)
    plt.title(title)
    plt.show()
    return predicted


predicted = evaluate(model, X_test, y_test, "random split")

About 87 % accuracy — against 76 % for a model that always says "yes". So the model did learn
something, but the headline number exaggerates it by a lot. Look at the confusion matrix: of the
molecules that genuinely do **not** cross the barrier, the model catches only around 60 %.

That matters, because "does not cross" is the class you actually want to detect when you are
filtering a library.

**What does this matrix mean?**

What's a False Positive?

What's a False Negative?

Which is worse for a brain drug?

The confusion matrix shows *where* the model loses points, not just how many.

* **True positive** — predicted "crosses", and it does.
* **True negative** — predicted "does not", and it does not.
* **False positive** — predicted "crosses", but it does not. A false alarm.
* **False negative** — predicted "does not", but it does. A missed opportunity.

**Which is worse for a brain drug?** The **false positive**. A false negative means you discard a
good molecule — a missed shot. A false positive means you select a molecule that cannot reach its
target, then spend months and a great deal of money synthesising and testing it before it fails
for a reason you could have predicted. A wasted shot costs more than a missed one.

## What did the model learn?

A random forest can report how much each descriptor contributed.

In [ ]:
importance = pd.DataFrame({
    "descriptor": DESCRIPTORS,
    "importance": model.feature_importances_,
}).sort_values("importance")

plt.barh(importance["descriptor"], importance["importance"], color="#4477aa")
plt.xlabel("importance")
plt.title("Which descriptors predict BBB penetration?")
plt.show()

**Looking at the plot, what is the most important feature?**

Does this make chemical sense?

Molecules that are highly polar (high TPSA), too water-soluble (low LogP) or to big (high MW) struggle to cross the fatty (lipid) barrier of the brain."

Compare this with Part 1: the model has rediscovered, from data alone, that polarity and
lipophilicity govern passive membrane permeation. That is a genuine QSAR result, and it is the
part of this exercise you could defend in a presentation.

The next exercise is the part you could not.

### Exercise 8 — Split the data like a medicinal chemist

A random split puts close analogues of the same series on both sides of the train/test line, so
the model can "recognise" a test molecule whose sibling it already memorised. Real use is never
like that: you apply a model to a **new series**. The standard imitation is a **scaffold split** —
group molecules by their Bemis–Murcko framework and keep any scaffold entirely on one side.

Use the helper below to build a new split, train the same model, and evaluate it with the same
function. Then compare the two sets of numbers.

```python
def scaffold_split(frame, test_fraction=0.2):
    """Split so that no Bemis-Murcko scaffold appears in both parts."""
    scaffolds = {}
    for i, mol in enumerate(frame["mol"]):
        core = MurckoScaffold.MurckoScaffoldSmiles(mol=mol, includeChirality=False)
        scaffolds.setdefault(core, []).append(i)

    groups = sorted(scaffolds.values(), key=len, reverse=True)
    n_test = int(len(frame) * test_fraction)
    test_rows, train_rows = [], []
    for group in groups:
        (test_rows if len(test_rows) + len(group) <= n_test else train_rows).extend(group)
    return train_rows, test_rows
```

<details>
<summary>🔍 Solution — and a result worth remembering</summary>

```python
train_rows, test_rows = scaffold_split(df)
scaffold_model = RandomForestClassifier(n_estimators=100, random_state=42)
scaffold_model.fit(X.iloc[train_rows], y.iloc[train_rows])

evaluate(scaffold_model, X.iloc[test_rows], y.iloc[test_rows], "scaffold split")
```

| | random split | scaffold split |
|---|---|---|
| accuracy | 0.87 | 0.83 |
| always-"yes" baseline | 0.77 | **0.86** |
| balanced accuracy | 0.77 | **0.57** |
| ROC-AUC | 0.90 | 0.67 |
| MCC | 0.61 | **0.18** |

Read the two columns carefully. **Accuracy hardly moved** — 0.87 to 0.83. On that number alone you
would conclude the model transfers fine to new chemistry.

**Everything else collapsed.** Balanced accuracy fell to 0.57, barely above a coin flip; MCC to
0.18. And 0.83 accuracy is now *below* the 0.86 you would get by always answering "yes": on new
scaffolds the model is **worse than useless**. Both are true at once because the scaffold test set
is more unbalanced — which is exactly why you never report a single number.

The general lesson:

1. **A random split overestimates performance on any dataset built from analogue series** — which
   is almost every medicinal chemistry dataset.
2. **Always quote a baseline.** A score without "what would guessing give?" is not a result.
3. **Validate the way the model will be used.** New scaffolds in production means new scaffolds in
   the test set.

</details>

### Exercise 9 — Look at what the model got wrong

Numbers tell you *how much* is wrong; structures tell you *what* is wrong. Pull out the molecules
the model misclassified with the most confidence and look at them.

<details>
<summary>🔍 Solution</summary>

```python
test_frame = df.loc[X_test.index].copy()
test_frame["predicted"] = predicted
test_frame["confidence"] = model.predict_proba(X_test)[:, 1]

wrong = test_frame[test_frame["predicted"] != test_frame["p_np"]]
worst = wrong.reindex((wrong["confidence"] - wrong["p_np"]).abs().sort_values().index[::-1]).head(6)

display(Draw.MolsToGridImage(
    list(worst["mol"]), molsPerRow=3, subImgSize=(260, 220),
    legends=[f"{n[:20]}\ntrue {t}, said {p} ({c:.2f})"
             for n, t, p, c in zip(worst["name"], worst["p_np"],
                                   worst["predicted"], worst["confidence"])]))
```

Typical things you will find, and all of them are the same kind of mistake:

* **Actively transported molecules.** Sugars, amino acids and some peptides cross the barrier
  through dedicated transporters despite being far too polar for passive diffusion. Our five
  descriptors describe passive permeation only, so these are predicted "no" and are in fact "yes".
* **Efflux substrates.** The opposite case: molecules that look perfectly greasy and permeable
  but are pumped straight back out by P-glycoprotein.
* **Very large or charged molecules** at the edge of the descriptor ranges, where the model is
  extrapolating.

None of these can be fixed by a better algorithm, because the information is not in the
descriptors. That is an **applicability domain** problem, and recognising it is worth more than
another percent of accuracy.

</details>

# Part 3 — Mapping chemical space

Five descriptors compress a molecule to five numbers. To *map* a whole library we want a richer
description, and then a way of seeing it.

For our QSAR model, we used 5 human-chosen descriptors.

To map 'all' of chemistry, we need a more general, high-dimensional representation.

We'll use Morgan Fingerprints (or ECFPs).

They convert each molecule into a long vector (e.g., 2048 numbers) that represents its substructures.

In [ ]:
morgan_generator = GetMorganGenerator(radius=2, fpSize=2048)

fingerprints = np.array([np.array(morgan_generator.GetFingerprint(m)) for m in df["mol"]])
print(f"fingerprint matrix: {fingerprints.shape[0]} molecules × {fingerprints.shape[1]} bits")
print(f"on average {fingerprints.sum(axis=1).mean():.0f} bits set per molecule "
      f"({fingerprints.mean()*100:.1f} % of the matrix)")

## Dimensionality reduction with UMAP

UMAP squeezes those 2048 columns down to two so we can look at them. One setting matters more
than all the others here:

**A fingerprint is a set of substructures, not a point in space.** The natural way to compare two
of them is the **Jaccard/Tanimoto** distance — the fraction of substructures they share — not the
Euclidean distance, which is dominated by the thousands of bits that are zero in both molecules.
Passing `metric="jaccard"` is a one-word change that makes the map measurably better organised.

In [ ]:
import umap

reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, metric="jaccard", random_state=42)

print("Fitting UMAP ...")
embedding = reducer.fit_transform(fingerprints)
df["UMAP1"], df["UMAP2"] = embedding[:, 0], embedding[:, 1]
print("done:", embedding.shape)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].scatter(df["UMAP1"], df["UMAP2"], s=6, alpha=0.5, color="grey")
axes[0].set_title("the chemical space of the dataset")

for value, color, label in [(1, "#d1495b", "crosses"), (0, "#4477aa", "does not cross")]:
    subset = df[df["p_np"] == value]
    axes[1].scatter(subset["UMAP1"], subset["UMAP2"], s=6, alpha=0.6, color=color, label=label)
axes[1].legend(markerscale=2)
axes[1].set_title("coloured by BBB penetration")

for ax in axes:
    ax.set_xlabel("UMAP 1")
    ax.set_ylabel("UMAP 2")
plt.tight_layout()
plt.show()

This is the 'Chemical Universe' of our dataset.

Each dot is a molecule.

Dots that are close together are structurally similar.

### Exercise 10 — Read the map

1. Are the two classes in separate regions, or mixed?
2. Find the **islands** — tight, well-separated clumps. What does an island mean chemically?
3. Connect this to Exercise 8: what does the island structure say about a *random* split?

<details>
<summary>🔍 Discussion</summary>

1. **Partly** separated: clearly red regions, clearly blue regions, and a large mixed zone — the
   geometric version of what the metrics said. A real but imperfect signal.
2. Each island is a **congeneric series**, many analogues of one scaffold. Public datasets are
   assembled from published papers, and papers report series, so the data arrives in clumps.
3. That is why the random split flattered the model: taking 20 % at random takes a slice out of
   *every* island and leaves the rest in training, so the model is asked about molecules whose
   neighbours it has memorised. The scaffold split holds out whole islands instead.

</details>

# Part 4 — Reading machines

Can a model pull facts out of a paper? This part uses **extractive question answering**: give the
model a block of text and a question, and it points at the span of that text which answers it. It
is the tool you would reach for to mine ADMET facts out of a literature corpus — and the point of
the exercise is to find out how far you can trust it.

### One warning first

`transformers` is the standard library for this, and in **version 5.0** the
`pipeline("question-answering")` interface used by most tutorials **was removed**. Code written
against version 4 now raises an error about an unknown task.

So instead of a pipeline we write the three lines it was hiding: tokenise the question together
with the context, run the model, read off which span it points at. This works on both versions,
and it shows what "extractive" literally means — the output is a **start and end position in your
text**. The model cannot say anything that is not already there.

In [ ]:
%pip install -q transformers torch

In [ ]:
import torch
import transformers
from transformers import AutoTokenizer, AutoModelForQuestionAnswering, AutoModelForSeq2SeqLM

transformers.logging.set_verbosity_error()      # the loading reports are noisy and harmless

EXTRACTIVE_MODEL = "distilbert/distilbert-base-cased-distilled-squad"
_loaded = {}


def ask(question, context, model_name=EXTRACTIVE_MODEL):
    """Extractive question answering: find the span of `context` that answers `question`."""
    if model_name not in _loaded:
        _loaded[model_name] = (AutoTokenizer.from_pretrained(model_name),
                               AutoModelForQuestionAnswering.from_pretrained(model_name))
    tokenizer, model = _loaded[model_name]

    inputs = tokenizer(question, context, return_tensors="pt", truncation=True, max_length=512)
    with torch.no_grad():
        output = model(**inputs)

    # the model scores every token for "is the answer starting / ending here?"
    start_scores = output.start_logits[0].softmax(-1)
    end_scores = output.end_logits[0].softmax(-1)
    start = int(start_scores.argmax())
    end = int(end_scores[start:].argmax()) + start

    answer = tokenizer.decode(inputs["input_ids"][0][start:end + 1], skip_special_tokens=True)
    return {"answer": answer.strip(), "score": float(start_scores[start] * end_scores[end])}


print("Extractive question answering ready.")

In [ ]:
context = """
Paracetamol, also known as acetaminophen, is a widely used over-the-counter medication.
It is well-absorbed, with an oral bioavailability of approximately 88%.
Metabolism occurs primarily in the liver, mainly through glucuronidation and sulfation.
A minor fraction is metabolized by CYP450 enzymes to a toxic intermediate, NAPQI.
At therapeutic doses, NAPQI is safely detoxified by glutathione.
However, in cases of overdose, glutathione is depleted, and NAPQI causes severe liver damage (hepatotoxicity).
"""

print(context)

Note:

This is an 'extractive' model.
It doesn't 'know' the answer, but it's an expert at finding the answer in a text.

However, imagine pointing this model at 10,000 research papers. You could ask, 'Which papers mention hERG toxicity?' in seconds."

### Exercise 11 — Predict before you run

Below are four questions. **Before running the cell**, write down for each one: what the answer
should be, and whether you expect the model to find it.

1. What is the oral bioavailability of Paracetamol?
2. Where is Paracetamol metabolized?
3. What is the main toxicity in case of overdose?
4. What is the name of the toxic intermediate?

In [ ]:
questions = [
    "What is the oral bioavailability of Paracetamol?",
    "Where is Paracetamol metabolized?",
    "What is the main toxicity in case of overdose?",
    "What is the name of the toxic intermediate?",
]

for question in questions:
    answer = ask(question, context)
    print(f"Q: {question}\nA: {answer['answer']}   (score {answer['score']:.2f})\n")

### What goes wrong, and why

Here is what we get (your scores may differ slightly with another version of the model):

| question | answer | score | |
|---|---|---|---|
| oral bioavailability? | 88 % | 0.82 | ✅ |
| **where** metabolized? | NAPQI | 0.53 | ❌ a *what*, not a *where* |
| main toxicity? | glutathione | 0.44 | ❌ that is the **antidote** |
| name of the toxic intermediate? | NAPQI | 0.97 | ✅ |

The two correct answers are the two whose wording appears almost literally in the text. The two
failures share a cause: **the model matches words that sit near each other, it does not model the
relationship between them.** "Metabolized" appears near "NAPQI", so NAPQI is offered for a *where*
question. "Toxicity" and "glutathione" appear in one sentence, so glutathione is offered — though
glutathione is what *prevents* the toxicity.

The useful part: **the confidence tracked the correctness** here, 0.82 and 0.97 against 0.53 and
0.44. Hold on to that for one more cell.

## How to Improve the Model

Here are three ways to get better answers, from a simple swap to a more advanced approach.

### 1. Use a Domain-Specific Extractive Model
This is the easiest and most direct improvement. The default model was trained on general text (news, Wikipedia). We can swap it with one trained specifically on biomedical and scientific papers.

These models will have a much better vocabulary and understanding of terms like "metabolite," "bioavailability," and "hepatotoxicity."

Examples (Hugging Face):

BioBERT: dmis-lab/biobert-base-cased-v1.1-squad (Trained on biomedical literature)

SciBERT: allenai/scibert_scivocab_cased (Trained on a broad corpus of scientific papers)

In [ ]:
for question in questions:
    answer = ask(question, context, model_name="dmis-lab/biobert-base-cased-v1.1-squad")
    print(f"Q: {question}\nA: {answer['answer']}   (score {answer['score']:.2f})\n")

### Exercise 12 — Did the domain-specific model actually help?

Compare the two tables before reading on. Would you deploy BioBERT instead of DistilBERT on this
evidence?

<details>
<summary>🔍 What we observed, and why it is a trap</summary>

| question | DistilBERT | BioBERT |
|---|---|---|
| bioavailability | 88 % (0.82) ✅ | 88 % (0.63) ✅ |
| **where** metabolized? | NAPQI (0.53) ❌ | CYP450 enzymes (0.04) ❌ |
| main toxicity? | glutathione (0.44) ❌ | **hepatotoxicity (0.14)** ✅ |
| toxic intermediate? | NAPQI (0.97) ✅ | NAPQI (0.98) ✅ |

The biomedical model fixed **one** of the two failures and still misses the "where" question. Two
conclusions, the second more important:

**1. "Use a domain-specific model" is a hypothesis, not a fix.** One fix out of four questions, on
one text, tells you nothing about which model is better. A real comparison needs a question set
with known answers and a score over all of them — which is more than many published comparisons
manage.

**2. Confidence scores are not comparable between models.** BioBERT's *correct* answer scored
**0.14**, far below the 0.7 threshold that worked for DistilBERT. A threshold tuned on one model is
meaningless on another.

</details>

### 2. Use an Abstractive Model (e.g., T5, BART)
The next step up is to use an abstractive model. Unlike extractive models, these models generate a new answer in their own words. They can synthesize information from multiple parts of the text.

This is a different task, often called text-generation or summarization, but you can frame the input to get a Q&A behavior.

How to implement (using T5):

In [ ]:
tokenizer_t5 = AutoTokenizer.from_pretrained("google/flan-t5-base")
model_t5 = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base")


def ask_generative(question, context):
    prompt = f"Answer the question using the context.\n\ncontext: {context}\n\nquestion: {question}"
    inputs = tokenizer_t5(prompt, return_tensors="pt", truncation=True, max_length=512)
    output = model_t5.generate(**inputs, max_new_tokens=40)
    return tokenizer_t5.decode(output[0], skip_special_tokens=True)


for question in questions:
    print(f"Q: {question}\nA: {ask_generative(question, context)}\n")

An abstractive model writes its own answer rather than pointing at a span, so it can phrase things
the text does not. In our run it got the bioavailability and NAPQI right and still missed both hard
questions ("CYP450 enzymes", "glutathione is depleted") — the same two failures, in nicer language.

That is the trade-off. An extractive model can only be wrong by **pointing at the wrong words**,
and it hands you the sentence so you can check it in seconds. A generative model can be wrong by
**writing something that was never in the text**, fluently and with no pointer back to a source.
For pulling numbers out of papers, the second failure mode is far more dangerous — and it is the
failure mode of every chatbot you will use for your project.

### 3. A full generative LLM

The step beyond: put the context into the prompt of a large generative model (Llama, Mistral,
Gemma) and ask it to reason. Retrieving the relevant text first and pasting it into the prompt is
**retrieval-augmented generation (RAG)**, and it is how most current literature assistants work.
It needs a GPU and more code than fits here — and, as above, it can be wrong far more fluently
than anything we ran.

### Exercise 13 — Point it at your own project

Paste the abstract of a paper about **your** drug–target pair as the context, and ask the questions
you actually need: the reported IC50 or Kd, which residues are involved, which resistance mutations
are mentioned, which assay was used.

Record two things for your presentation: **how often it is right** (check every answer against the
abstract yourself), and **where it fails** — numbers with units, relationships across two
sentences, and anything needing a comparison are the usual weak points.

In [ ]:
my_abstract = """
Paste the abstract of a paper about your target here.
"""

my_questions = [
    "What is the binding affinity?",
    "Which mutation causes resistance?",
]

for question in my_questions:
    answer = ask(question, my_abstract)
    print(f"Q: {question}\nA: {answer['answer']}   (score {answer['score']:.2f})\n")

# Taking this to your project

| from this notebook | use it for |
|---|---|
| `show_features`, `feature_counts` | describe your ligand and its analogues in feature terms |
| `common_pharmacophore`, `as_query` | derive a model from your actives instead of guessing one |
| `matches_pharmacophore`, `match_report` | test it — and check *which atoms* matched |
| `pubchem_similar`, `screen` | build a reproducible shortlist of compounds to dock |
| scaffold split + baselines | any model you build, and any model you cite |
| Morgan fingerprints + UMAP | show where your compounds sit in chemical space |
| QA pipeline | a first pass through the literature, never the last |

Three honest limitations — candidates for your required "three reasons our conclusion could be
wrong" slide:

1. **A pharmacophore is necessary, not sufficient.** It cannot separate two molecules with the
   same features and different shapes — exactly the situation in most resistance cases.
2. **A model validated on a random split says nothing about new chemistry.** Check how the models
   you cite were split before quoting their accuracy.
3. **Neither method knows about the protein.** Transport, efflux, conformational selection and
   covalent chemistry are all invisible here. That is what structures and docking are for — next.